# scChromatic validation: executable protocol and analysis scaffold

**Status:** reusable protocol, not completed evidence. This notebook prepares four validation pillars: real single-cell datasets, comparator benchmarks, method ablations, and a participant-study feasibility pilot. The synthetic package dataset is used only to verify that the analysis plumbing runs.

The notebook is deliberately non-installing and non-writing by default. Real-data downloads, the longer benchmark, optional Polychrome generation, and artifact export each require an explicit flag below.

## Pre-specified questions, outcomes, and claim boundaries

1. **Relationship fit:** do color distances track independently fixed label affinities? Report inverse-affinity Spearman correlation and the distance margin between low- and high-affinity label pairs.
2. **Accessibility diagnostics:** report minimum CIEDE2000 separation under normal, deutan, protan, and tritan simulations. Simulation is a stress test, not proof of universal accessibility.
3. **Persistence:** after labels are subsetted or reordered, what fraction of retained labels changes color?
4. **Stability/performance trade-off:** how do the canonical map, stability budget, CVD objective, sample aggregation, and random seed change the results?
5. **Human feasibility:** can participants complete label lookup, cross-panel tracking, related-pair, and dense/rare-label tasks with acceptable burden and data completeness? The pilot estimates feasibility only; it is not powered to establish superiority.

Freeze dataset exclusions, eligible labels, primary outcomes, comparator capacities, seeds, and progression criteria before the confirmatory run. Do not select the best seed or palette after inspecting test results.

In [ ]:
SEED <- 20260804L
RUN_REAL_DATA <- FALSE
RUN_SCALE_STRESS <- FALSE
RUN_FULL_BENCHMARK <- FALSE
RUN_POLYCHROME_GENERATION <- FALSE
WRITE_OUTPUTS <- FALSE
ANCHOR_LABELS <- character()  # Set and freeze biologically canonical hard locks here.
EXCLUDED_LABELS <- c("", "unknown", "unclassified", "unclear", "n/a")

find_repo_root <- function(start = getwd()) {
  path <- normalizePath(start, winslash = "/", mustWork = TRUE)
  repeat {
    if (file.exists(file.path(path, "DESCRIPTION")) &&
        dir.exists(file.path(path, "R"))) return(path)
    parent <- dirname(path)
    if (identical(parent, path)) return(NA_character_)
    path <- parent
  }
}

repo_root <- find_repo_root()
if (!is.na(repo_root) && requireNamespace("pkgload", quietly = TRUE)) {
  pkgload::load_all(repo_root, quiet = TRUE, helpers = FALSE)
} else {
  library(scChromatic)
}
stopifnot(
  requireNamespace("farver", quietly = TRUE),
  requireNamespace("ggplot2", quietly = TRUE),
  requireNamespace("scales", quietly = TRUE),
  requireNamespace("colorspace", quietly = TRUE)
)
set.seed(SEED)

packages_to_record <- c(
  "scChromatic", "R", "farver", "colorspace", "ggplot2",
  "scales", "scRNAseq", "scater", "scuttle", "Polychrome"
)
package_versions <- data.frame(
  package = packages_to_record,
  version = vapply(packages_to_record, function(pkg) {
    if (pkg == "R") return(as.character(getRversion()))
    if (!requireNamespace(pkg, quietly = TRUE)) return(NA_character_)
    as.character(utils::packageVersion(pkg))
  }, character(1)),
  stringsAsFactors = FALSE
)
package_versions

## Real-data and comparator manifest

The confirmatory pair is Baron plus Segerstolpe human pancreas: both support multi-donor analysis, while using different studies reduces dependence on one annotation pipeline. Darmanis provides a small cross-tissue run and pilot-stimulus source. Zilionis is opt-in as a scale stress test. Accessors and metadata must be checked after every data-package update.

Main relationship graphs use PCA or another pre-specified scientific latent space. UMAP is reserved for display and an explicit coordinate-choice ablation because its plotted distances are not treated as quantitative biological distances.

In [ ]:
dataset_manifest <- data.frame(
  dataset_id = c("baron_human", "segerstolpe", "darmanis_brain", "zilionis_lung"),
  role = c("confirmatory", "confirmatory", "cross-tissue/pilot", "scale stress"),
  accessor = c(
    "scRNAseq::BaronPancreasData('human')",
    "scRNAseq::SegerstolpePancreasData()",
    "scRNAseq::DarmanisBrainData()",
    "scRNAseq::ZilionisLungData('human')"
  ),
  label_candidates = c(
    "label", "cell type|cell_type", "cell.type|cell_type|cell type",
    "Major cell type|Major.cell.type|major_cell_type"
  ),
  sample_candidates = c(
    "donor", "individual", "individual|patient|sample|donor", "Patient|patient"
  ),
  max_cells = c(5000L, 5000L, 5000L, 5000L),
  min_cells_per_sample_label = c(20L, 20L, 1L, 20L),
  min_samples_per_label = c(2L, 2L, 1L, 2L),
  source_url = c(
    "https://bioconductor.org/packages/release/data/experiment/vignettes/scRNAseq/inst/doc/scRNAseq.html",
    "https://bioconductor.org/books/release/OSCA.workflows/segerstolpe-human-pancreas-smart-seq2.html",
    "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE67835",
    "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE127465"
  ),
  stringsAsFactors = FALSE
)

comparator_manifest <- data.frame(
  method_family = c(
    "scChromatic context optimizer", "registered fixed palettes",
    "ggplot2 hue", "colorspace qualitative HCL",
    "Polychrome generated (optional)"
  ),
  methods = c(
    "canonical stability budget; unconstrained map",
    "chromatic; ditto40; glasbey32; polychrome36; okabe_ito; tol_muted; d3_rainbow",
    "scales::hue_pal", "colorspace::qualitative_hcl('Dark 3')",
    "Polychrome::createPalette(target='deuteranope')"
  ),
  capacity_policy = c(
    "candidate grid; canonical extension disclosed if >40 labels",
    "native capacity only; no extension", "algorithmic n", "algorithmic n",
    "algorithmic n; fixed seed and M"
  ),
  source_url = c(
    "https://github.com/xie186/scChromatic",
    "https://xie186.github.io/scChromatic/articles/palette-gallery.html",
    "https://ggplot2.tidyverse.org/reference/scale_hue.html",
    "https://colorspace.r-forge.r-project.org/articles/hcl_palettes.html",
    "https://r-forge.r-universe.dev/Polychrome/doc/manual.html"
  ),
  stringsAsFactors = FALSE
)
dataset_manifest
comparator_manifest

### Prepared-table contract and optional Bioconductor loader

For a frozen analysis, prefer one reviewed RDS table per dataset with columns `dataset_id`, `cell_id`, `label`, `sample`, and `PC1` through `PC10`. This decouples validation from moving upstream object schemas. The optional loader below creates that table but intentionally does not add Bioconductor packages to scChromatic's package dependencies. Install them separately only for validation.

Before subsampling, the confirmatory datasets require at least 20 cells in at least two samples for a label to be eligible; Darmanis uses a one-cell/one-sample exploratory rule. These thresholds are visible in the manifest and must be frozen before analysis. Subsampling then occurs before PCA, is seeded, retains at least one cell from every eligible sample-by-label stratum, and otherwise samples from the remaining cells. Record the resulting table checksum in the study archive.

In [ ]:
dataset_fields <- list(
  baron_human = list(label = c("label"), sample = c("donor")),
  segerstolpe = list(label = c("cell type", "cell_type"), sample = c("individual")),
  darmanis_brain = list(
    label = c("cell.type", "cell_type", "cell type"),
    sample = c("individual", "patient", "sample", "donor")
  ),
  zilionis_lung = list(
    label = c("Major cell type", "Major.cell.type", "major_cell_type"),
    sample = c("Patient", "patient")
  )
)

normalize_field_name <- function(x) tolower(gsub("[^[:alnum:]]", "", x))

pick_metadata_field <- function(metadata, candidates, required = TRUE) {
  available <- names(metadata)
  hits <- match(normalize_field_name(candidates), normalize_field_name(available), nomatch = 0L)
  hits <- hits[hits > 0L]
  if (length(hits)) return(available[hits[[1L]]])
  if (required) {
    stop("None of the pre-specified metadata fields were found: ",
         paste(candidates, collapse = ", "))
  }
  NULL
}

stratified_subsample <- function(label, sample_id, n_max, seed) {
  index <- seq_along(label)
  if (length(index) <= n_max) return(index)
  groups <- split(index, interaction(sample_id, label, drop = TRUE, lex.order = TRUE))
  if (length(groups) > n_max) stop("n_max is smaller than the observed sample-by-label strata.")
  set.seed(seed)
  retained <- vapply(groups, function(x) x[[sample.int(length(x), 1L)]], integer(1))
  pool <- setdiff(index, retained)
  extra_n <- n_max - length(retained)
  extra <- if (extra_n) pool[sample.int(length(pool), extra_n)] else integer()
  sort(c(retained, extra))
}

fetch_real_sce <- function(dataset_id) {
  if (!requireNamespace("scRNAseq", quietly = TRUE)) {
    stop("Install Bioconductor package 'scRNAseq' to enable real-data downloads.")
  }
  switch(
    dataset_id,
    baron_human = scRNAseq::BaronPancreasData("human"),
    segerstolpe = scRNAseq::SegerstolpePancreasData(),
    darmanis_brain = scRNAseq::DarmanisBrainData(),
    zilionis_lung = scRNAseq::ZilionisLungData("human"),
    stop("Unknown dataset_id: ", dataset_id)
  )
}

prepare_sce <- function(sce, dataset_id, max_cells = 5000L, seed = SEED) {
  required <- c("SummarizedExperiment", "SingleCellExperiment", "scuttle", "scater")
  missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
  if (length(missing)) stop("Install validation dependencies: ", paste(missing, collapse = ", "))
  fields <- dataset_fields[[dataset_id]]
  if (is.null(fields)) stop("No field specification for ", dataset_id)
  metadata <- as.data.frame(SummarizedExperiment::colData(sce))
  label_field <- pick_metadata_field(metadata, fields$label)
  sample_field <- pick_metadata_field(metadata, fields$sample, required = FALSE)
  label <- trimws(as.character(metadata[[label_field]]))
  sample_id <- if (is.null(sample_field)) rep(dataset_id, length(label)) else
    trimws(as.character(metadata[[sample_field]]))
  valid <- !is.na(label) & nzchar(label) &
    !tolower(label) %in% tolower(EXCLUDED_LABELS) &
    !is.na(sample_id) & nzchar(sample_id)
  sce <- sce[, valid]
  label <- label[valid]
  sample_id <- sample_id[valid]
  protocol <- dataset_manifest[match(dataset_id, dataset_manifest$dataset_id), ]
  support <- table(label, sample_id)
  supported_samples <- rowSums(support >= protocol$min_cells_per_sample_label[[1L]])
  eligible_labels <- names(supported_samples)[
    supported_samples >= protocol$min_samples_per_label[[1L]]
  ]
  label_support <- data.frame(
    dataset_id = dataset_id, label = rownames(support), cells = rowSums(support),
    supported_samples = unname(supported_samples),
    min_cells_per_sample_label = protocol$min_cells_per_sample_label[[1L]],
    min_samples_per_label = protocol$min_samples_per_label[[1L]],
    eligible = rownames(support) %in% eligible_labels, stringsAsFactors = FALSE
  )
  if (length(eligible_labels) < 2L) stop("Fewer than two labels pass the pre-specified support rule.")
  eligible <- label %in% eligible_labels
  sce <- sce[, eligible]
  label <- label[eligible]
  sample_id <- sample_id[eligible]
  keep <- stratified_subsample(label, sample_id, max_cells, seed)
  sce <- sce[, keep]
  label <- label[keep]
  sample_id <- sample_id[keep]
  if (length(unique(label)) < 2L) stop("At least two eligible labels are required.")
  sce <- scuttle::logNormCounts(sce)
  n_pc <- min(10L, ncol(sce) - 1L, nrow(sce) - 1L)
  if (n_pc < 2L) stop("The filtered object is too small for PCA.")
  set.seed(seed)
  sce <- scater::runPCA(sce, ncomponents = n_pc, ntop = min(2000L, nrow(sce)))
  pcs <- SingleCellExperiment::reducedDim(sce, "PCA")
  colnames(pcs) <- paste0("PC", seq_len(ncol(pcs)))
  cell_id <- colnames(sce)
  if (is.null(cell_id)) cell_id <- sprintf("%s_cell_%06d", dataset_id, seq_len(ncol(sce)))
  cell_id <- make.unique(as.character(cell_id))
  prepared <- data.frame(
    dataset_id = dataset_id, cell_id = cell_id, label = label, sample = sample_id,
    pcs, check.names = FALSE, stringsAsFactors = FALSE
  )
  attr(prepared, "label_support") <- label_support
  prepared
}

validate_prepared_table <- function(x) {
  required <- c("dataset_id", "cell_id", "label", "sample")
  if (!all(required %in% names(x))) stop("Prepared table is missing required columns.")
  pc <- grep("^PC[0-9]+$", names(x), value = TRUE)
  if (length(pc) < 2L) stop("Prepared table requires at least PC1 and PC2.")
  if (anyDuplicated(x$cell_id)) stop("cell_id must be unique.")
  if (any(!is.finite(as.matrix(x[, pc, drop = FALSE])))) stop("PC values must be finite.")
  if (length(unique(x$label)) < 2L) stop("At least two labels are required.")
  x
}

read_prepared_rds <- function(path) validate_prepared_table(readRDS(path))

## Benchmark functions

All fixed palettes are evaluated at native capacity with `extend = 'error'`. Generic color sets use both their natural assignment and seeded label permutations later, so palette quality is not conflated with a favorable label ordering. Every method is scored against the same frozen relationship matrix.

`final_objective` is scChromatic's own construction objective and is therefore method-concordant rather than a neutral external endpoint. The rank correlation, high/low-affinity margin, CVD diagnostics, persistence, and runtime are reported alongside it. No single metric is declared a universal winner.

In [ ]:
timed_value <- function(f) {
  started <- proc.time()[["elapsed"]]
  value <- f()
  list(value = value, seconds = unname(proc.time()[["elapsed"]] - started))
}

named_for_labels <- function(colors, labels) {
  if (length(colors) != length(labels)) stop("One color per label is required.")
  stats::setNames(unname(colors), labels)
}

register_method <- function(value, seconds, family, assignment, source_url, labels) {
  colors <- if (inherits(value, "sc_color_map")) as_named_colors(value) else value
  colors <- named_for_labels(colors[labels], labels)
  list(
    colors = colors, object = value, seconds = seconds, family = family,
    assignment = assignment, source_url = source_url
  )
}

build_methods <- function(labels, affinity, seed = SEED) {
  labels <- rownames(affinity)
  n <- length(labels)
  methods <- list()
  stored_ids <- c(
    "chromatic", "ditto40", "glasbey32", "polychrome36",
    "okabe_ito", "tol_muted", "d3_rainbow"
  )
  for (id in stored_ids) {
    info <- sc_palette_info(id)
    if (n > info$max_n[[1L]]) next
    result <- timed_value(function() named_for_labels(
      sc_palette(id, n = n, extend = "error"), labels
    ))
    methods[[paste0("registered_", id)]] <- register_method(
      result$value, result$seconds,
      if (id == "d3_rainbow") "compatibility negative control" else "registered palette",
      "generic", info$source_url[[1L]], labels
    )
  }
  result <- timed_value(function() named_for_labels(scales::hue_pal()(n), labels))
  methods$ggplot_hue <- register_method(
    result$value, result$seconds, "external comparator", "generic",
    "https://ggplot2.tidyverse.org/reference/scale_hue.html", labels
  )
  result <- timed_value(function() named_for_labels(
    colorspace::qualitative_hcl(n, palette = "Dark 3"), labels
  ))
  methods$colorspace_dark3 <- register_method(
    result$value, result$seconds, "external comparator", "generic",
    "https://colorspace.r-forge.r-project.org/articles/hcl_palettes.html", labels
  )
  if (RUN_POLYCHROME_GENERATION) {
    if (!requireNamespace("Polychrome", quietly = TRUE)) {
      stop("RUN_POLYCHROME_GENERATION requires the Polychrome package.")
    }
    set.seed(seed)
    result <- timed_value(function() named_for_labels(
      Polychrome::createPalette(
        n, seedcolors = c("#006BA4", "#FF800E"),
        target = "deuteranope", M = 50000
      ), labels
    ))
    methods$polychrome_generated_deutan <- register_method(
      result$value, result$seconds, "external generated comparator", "generic",
      "https://r-forge.r-universe.dev/Polychrome/doc/manual.html", labels
    )
  }
  canonical_extend <- if (n <= sc_palette_info("chromatic")$max_n[[1L]]) "error" else "generate"
  canonical <- named_for_labels(
    sc_palette("chromatic", n = n, extend = canonical_extend), labels
  )
  budget <- min(n, max(1L, floor(0.20 * n)))
  locked <- intersect(ANCHOR_LABELS, labels)
  result <- timed_value(function() sc_relationship_map(
    affinity, canonical = canonical, locked = locked,
    stability_budget = budget, seed = seed
  ))
  methods$scChromatic_context_stable <- register_method(
    result$value, result$seconds, "proposed method", "context optimized",
    "https://github.com/xie186/scChromatic", labels
  )
  result <- timed_value(function() sc_relationship_map(affinity, seed = seed))
  methods$scChromatic_context_free <- register_method(
    result$value, result$seconds, "proposed method ablation", "context optimized",
    "https://github.com/xie186/scChromatic", labels
  )
  methods
}

score_colors <- function(method, colors, reference_affinity, seconds = NA_real_,
                         family = NA_character_, assignment = NA_character_,
                         dataset_id = NA_character_) {
  labels <- rownames(reference_affinity)
  colors <- named_for_labels(colors[labels], labels)
  audit <- sc_palette_audit(colors)
  frozen <- sc_relationship_map(
    reference_affinity, canonical = colors, stability_budget = 0L, seed = SEED
  )
  lab <- farver::decode_colour(unname(colors), to = "lab")
  distance <- farver::compare_colour(lab, lab, from_space = "lab", method = "cie2000")
  upper <- upper.tri(reference_affinity)
  affinity <- as.numeric(reference_affinity[upper])
  color_distance <- as.numeric(distance[upper])
  q <- stats::quantile(affinity, c(0.25, 0.75), names = FALSE, type = 2)
  rank_fit <- if (stats::sd(affinity) == 0 || stats::sd(color_distance) == 0) NA_real_ else
    stats::cor(1 - affinity, color_distance, method = "spearman")
  margin <- stats::median(color_distance[affinity <= q[[1L]]]) -
    stats::median(color_distance[affinity >= q[[2L]]])
  vision <- audit$vision
  data.frame(
    dataset_id = dataset_id, method = method, family = family, assignment = assignment,
    label_count = length(labels), inverse_affinity_spearman = rank_fit,
    unrelated_related_margin = margin,
    min_normal_cie2000 = vision$min_distance[vision$vision == "none"],
    min_worst_cvd_cie2000 = min(vision$min_distance[vision$vision != "none"], na.rm = TRUE),
    min_contrast = audit$summary$min_contrast,
    duplicate_count = audit$summary$duplicate_count,
    sc_objective = frozen$context$final_objective,
    sc_normal_fit_stress = frozen$context$normal_fit_stress,
    sc_worst_cvd_shortfall = frozen$context$worst_cvd_shortfall,
    construction_seconds = seconds, stringsAsFactors = FALSE
  )
}

score_method_set <- function(methods, reference_affinity, dataset_id) {
  do.call(rbind, lapply(names(methods), function(id) {
    x <- methods[[id]]
    score_colors(
      id, x$colors, reference_affinity, x$seconds, x$family,
      x$assignment, dataset_id
    )
  }))
}

relationship_summary <- function(affinity, dataset_id) {
  context <- attr(affinity, "sc_context")
  support <- context$pair_support$sample_count
  data.frame(
    dataset_id = dataset_id, labels = nrow(affinity), cells = context$cell_count,
    samples = context$sample_count, k = context$k_requested,
    zero_filled_pairs = sum(support == 0L),
    minimum_pair_support = min(support),
    context_md5 = context$context_md5, stringsAsFactors = FALSE
  )
}

## Offline smoke test (synthetic; no external-validity claim)

`sc_example` is deterministic synthetic PBMC-like data. Its display coordinates are used here only to exercise every benchmark pathway quickly. Results from this section must not appear as biological validation or manuscript performance evidence.

In [ ]:
load_smoke_data <- function() {
  target <- new.env(parent = emptyenv())
  if (!is.na(repo_root) && file.exists(file.path(repo_root, "data", "sc_example.rda"))) {
    load(file.path(repo_root, "data", "sc_example.rda"), envir = target)
  } else {
    utils::data("sc_example", package = "scChromatic", envir = target)
  }
  target$sc_example
}

smoke_raw <- load_smoke_data()
smoke_data <- data.frame(
  dataset_id = "synthetic_smoke_only", cell_id = smoke_raw$cell_id,
  label = as.character(smoke_raw$cell_type), sample = as.character(smoke_raw$sample),
  PC1 = smoke_raw$UMAP1, PC2 = smoke_raw$UMAP2, stringsAsFactors = FALSE
)
smoke_affinity <- sc_relationship_from_knn(
  smoke_data[, c("PC1", "PC2")], smoke_data$label, smoke_data$sample,
  k = 15L, aggregate = "mean", unobserved = "error"
)
smoke_methods <- build_methods(rownames(smoke_affinity), smoke_affinity)
smoke_benchmark <- score_method_set(
  smoke_methods, smoke_affinity, "synthetic_smoke_only"
)
smoke_relationship_summary <- relationship_summary(smoke_affinity, "synthetic_smoke_only")
smoke_relationship_summary
smoke_benchmark[order(-smoke_benchmark$inverse_affinity_spearman), ]

In [ ]:
ggplot2::ggplot(
  smoke_benchmark,
  ggplot2::aes(
    x = inverse_affinity_spearman, y = min_worst_cvd_cie2000,
    color = family, label = method
  )
) +
  ggplot2::geom_point(size = 2) +
  ggplot2::geom_text(check_overlap = TRUE, nudge_y = 0.5, size = 3, show.legend = FALSE) +
  ggplot2::labs(
    title = "Synthetic smoke test only",
    x = "Relationship fit (higher is better)",
    y = "Minimum CIEDE2000 across simulated CVD views"
  ) +
  ggplot2::theme_minimal(base_size = 11)

## Assignment sensitivity and persistence stress

A generic palette is not a complete label-color method: its apparent relationship fit can change when colors are assigned to labels in another order. Report the natural order and a seeded distribution of assignments (at least 100 permutations for the manuscript). Separately, compare a full persistent named map with positional reapplication after subset/reorder operations.

In [ ]:
assignment_sensitivity <- function(methods, affinity, dataset_id,
                                   n_permutations = if (RUN_FULL_BENCHMARK) 100L else 3L,
                                   seed = SEED) {
  labels <- rownames(affinity)
  generic <- names(methods)[vapply(methods, function(x) x$assignment == "generic", logical(1))]
  set.seed(seed)
  rows <- list()
  out_i <- 0L
  for (id in generic) {
    base <- unname(methods[[id]]$colors[labels])
    assignments <- c(list(natural = base), lapply(seq_len(n_permutations), function(i) {
      base[sample.int(length(base))]
    }))
    names(assignments)[-1L] <- sprintf("permuted_%03d", seq_len(n_permutations))
    for (assignment_id in names(assignments)) {
      out_i <- out_i + 1L
      scored <- score_colors(
        id, named_for_labels(assignments[[assignment_id]], labels), affinity,
        family = methods[[id]]$family, assignment = "assignment sensitivity",
        dataset_id = dataset_id
      )
      scored$assignment_id <- assignment_id
      rows[[out_i]] <- scored
    }
  }
  do.call(rbind, rows)
}

persistence_stress <- function(full_map, dataset_id = NA_character_, palette_id = "chromatic",
                               n_reps = 25L, seed = SEED) {
  if (!inherits(full_map, "sc_color_map")) stop("full_map must be an sc_color_map.")
  full_colors <- as_named_colors(full_map)
  labels <- names(full_colors)
  positional_full <- named_for_labels(
    sc_palette(palette_id, length(labels), extend = "error"), labels
  )
  set.seed(seed)
  do.call(rbind, lapply(seq_len(n_reps), function(i) {
    retained_n <- sample(seq.int(max(2L, floor(length(labels) / 2)), length(labels)), 1L)
    retained <- sample(labels, retained_n, replace = FALSE)
    persistent_subset <- as_named_colors(full_map[retained])
    positional_subset <- named_for_labels(
      sc_palette(palette_id, retained_n, extend = "error"), retained
    )
    data.frame(
      dataset_id = dataset_id, replicate = i,
      method = c("persistent_named_map", "positional_reapplication"),
      retained_labels = retained_n,
      recolored_fraction = c(
        mean(full_colors[retained] != persistent_subset[retained]),
        mean(positional_full[retained] != positional_subset[retained])
      ),
      stringsAsFactors = FALSE
    )
  }))
}

smoke_assignment_sensitivity <- assignment_sensitivity(
  smoke_methods, smoke_affinity, "synthetic_smoke_only"
)
smoke_primary_map <- smoke_methods$scChromatic_context_stable$object
smoke_persistence <- persistence_stress(smoke_primary_map, "synthetic_smoke_only")
aggregate(inverse_affinity_spearman ~ method, smoke_assignment_sensitivity,
          function(x) c(min = min(x), median = stats::median(x), max = max(x)))
aggregate(recolored_fraction ~ method, smoke_persistence, mean)

## Ablations

The fixed reference is the sample-balanced mean kNN affinity. Each ablation changes one construction choice, but every resulting color map is evaluated on that same reference. The planned variants are: no context/canonical only, pooled cells, median sample aggregation, normal vision only, no canonical map, zero/full stability budget, and alternate fixed seeds. Shape and pattern allocation are reported as redundant-encoding outcomes because they do not change the colors.

For real data, `unobserved = 'zero'` is an explicit assumption that labels never jointly observed in a sample have zero affinity. The number of such pairs and pairwise sample support are mandatory outputs; repeat the analysis after restricting to adequately supported pairs.

In [ ]:
derive_affinity <- function(data, sample_balanced = TRUE, aggregate = "mean",
                            unobserved = "zero", k = 15L) {
  pc <- grep("^PC[0-9]+$", names(data), value = TRUE)
  sc_relationship_from_knn(
    data[, pc, drop = FALSE], data$label,
    if (sample_balanced) data$sample else NULL,
    k = k, aggregate = aggregate, unobserved = unobserved
  )
}

run_ablations <- function(data, dataset_id, seed = SEED, unobserved = "zero") {
  reference <- derive_affinity(data, TRUE, "mean", unobserved)
  pooled <- derive_affinity(data, FALSE, "mean", unobserved)
  median_affinity <- derive_affinity(data, TRUE, "median", unobserved)
  labels <- rownames(reference)
  extend <- if (length(labels) <= sc_palette_info("chromatic")$max_n[[1L]]) "error" else "generate"
  canonical <- named_for_labels(sc_palette("chromatic", length(labels), extend = extend), labels)
  budget <- min(length(labels), max(1L, floor(0.20 * length(labels))))
  locked <- intersect(ANCHOR_LABELS, labels)
  fits <- list()
  fits$no_context_canonical <- list(value = canonical, seconds = 0)
  fit <- function(affinity, canonical_arg = canonical, budget_arg = budget,
                  cvd = c("deutan", "protan", "tritan"), seed_arg = seed) {
    timed_value(function() sc_relationship_map(
      affinity, canonical = canonical_arg, locked = if (is.null(canonical_arg)) character() else locked,
      stability_budget = budget_arg, seed = seed_arg, cvd = cvd
    ))
  }
  fits$full <- fit(reference)
  fits$pooled_cells <- fit(pooled)
  fits$median_samples <- fit(median_affinity)
  fits$normal_vision_only <- fit(reference, cvd = "none")
  fits$no_canonical <- fit(reference, canonical_arg = NULL, budget_arg = 0L)
  fits$stability_budget_0 <- fit(reference, budget_arg = 0L)
  fits$stability_budget_full <- fit(reference, budget_arg = length(labels))
  fits$seed_17 <- fit(reference, seed_arg = 17L)
  fits$seed_186 <- fit(reference, seed_arg = 186L)
  rows <- do.call(rbind, lapply(names(fits), function(id) {
    value <- fits[[id]]$value
    colors <- if (inherits(value, "sc_color_map")) as_named_colors(value) else value
    score_colors(
      id, colors, reference, fits[[id]]$seconds, "ablation",
      "fixed reference", dataset_id
    )
  }))
  full_colors <- as_named_colors(fits$full$value)
  encodings <- list(
    shape = sc_redundant_encoding(full_colors, channel = "shape"),
    pattern = sc_redundant_encoding(full_colors, channel = "pattern")
  )
  encoding_summary <- do.call(rbind, lapply(names(encodings), function(channel) {
    x <- encodings[[channel]]
    data.frame(
      dataset_id = dataset_id, channel = channel, colors_changed = FALSE,
      encoding_groups = max(x$encoding_group),
      conflict_pairs = nrow(attr(x, "conflicts")), stringsAsFactors = FALSE
    )
  }))
  list(
    reference = reference, maps = fits, scores = rows,
    encodings = encodings, encoding_summary = encoding_summary
  )
}

smoke_ablations <- run_ablations(
  smoke_data, "synthetic_smoke_only", unobserved = "error"
)
smoke_ablations$scores[order(-smoke_ablations$scores$inverse_affinity_spearman), ]
smoke_ablations$encoding_summary

## Opt-in real-data run

Before setting `RUN_REAL_DATA <- TRUE`, inspect the current `colData()` fields, freeze the eligible label list, confirm donor/sample semantics, and archive prepared-table checksums. Baron and Segerstolpe are the confirmatory datasets; Darmanis is exploratory. Zilionis runs only when `RUN_SCALE_STRESS` is also true.

The cell cap is a pre-specified runtime control, not a request to cherry-pick cells. Run a sample-size ladder for the scale study and report wall time, peak memory outside this notebook, label coverage, and whether the exact-kNN implementation completed.

In [ ]:
run_validation_dataset <- function(data, dataset_id) {
  data <- validate_prepared_table(data)
  label_support <- attr(data, "label_support")
  if (is.null(label_support)) {
    label_support <- data.frame(
      dataset_id = dataset_id, label = names(table(data$label)),
      cells = as.integer(table(data$label)), supported_samples = NA_integer_,
      min_cells_per_sample_label = NA_integer_, min_samples_per_label = NA_integer_,
      eligible = TRUE, stringsAsFactors = FALSE
    )
  }
  affinity <- derive_affinity(data, TRUE, "mean", "zero")
  methods <- build_methods(rownames(affinity), affinity)
  list(
    data_summary = data.frame(
      dataset_id = dataset_id, cells = nrow(data), labels = length(unique(data$label)),
      samples = length(unique(data$sample)), stringsAsFactors = FALSE
    ),
    relationship_summary = relationship_summary(affinity, dataset_id),
    label_support = label_support, prepared = data, affinity = affinity, methods = methods,
    benchmark = score_method_set(methods, affinity, dataset_id),
    assignment_sensitivity = assignment_sensitivity(methods, affinity, dataset_id),
    persistence = persistence_stress(methods$scChromatic_context_stable$object, dataset_id),
    ablations = run_ablations(data, dataset_id, unobserved = "zero")
  )
}

real_results <- list()
if (RUN_REAL_DATA) {
  selected <- c("baron_human", "segerstolpe", "darmanis_brain")
  if (RUN_SCALE_STRESS) selected <- c(selected, "zilionis_lung")
  for (i in seq_along(selected)) {
    id <- selected[[i]]
    max_cells <- dataset_manifest$max_cells[match(id, dataset_manifest$dataset_id)]
    prepared <- prepare_sce(fetch_real_sce(id), id, max_cells, SEED + i)
    real_results[[id]] <- run_validation_dataset(prepared, id)
  }
}
if (length(real_results)) {
  do.call(rbind, lapply(real_results, `[[`, "data_summary"))
  do.call(rbind, lapply(real_results, `[[`, "relationship_summary"))
} else {
  message("Real-data run is disabled. Set RUN_REAL_DATA <- TRUE after protocol review.")
}

## Participant-study feasibility pilot

**Purpose:** test recruitment, task comprehension, rendering, session duration, missingness, and outcome capture before a powered study. The default 12 coded participants below are a logistics placeholder, not a justified sample size. Choose the final number from the desired precision of feasibility estimates with a statistician, then pre-register it.

**Within-participant conditions:** (A) positional/reapplied color baseline, (B) persistent canonical color, (C) persistent context-aware color, and (D) the same persistent context-aware color plus redundant shape. This separates persistence, relationship-aware assignment, and redundant encoding. Pattern can replace shape in a separately pre-specified accessibility cohort; do not add conditions after seeing results. Use two frozen datasets and four tasks: label lookup, cross-panel tracking, related-pair judgment, and dense/rare-label detection. Accuracy is the primary task outcome; response time and confidence are secondary.

**Safeguards:** obtain institutional ethics/IRB determination and consent before recruitment; use coded IDs only; keep the condition key separate; collect no names or emails in these files; provide breaks and an accessible response mode; do not diagnose color-vision status; and do not treat CVD simulations as a replacement for participation by people with color-vision differences. Pilot summaries are descriptive, with uncertainty intervals where appropriate, and are not confirmatory efficacy tests.

In [ ]:
make_pilot_manifest <- function(n_participants = 12L, seed = SEED) {
  if (n_participants < 1L) stop("n_participants must be positive.")
  participants <- sprintf("P%03d", seq_len(n_participants))
  condition_ids <- c(
    "positional_baseline", "persistent_canonical_color",
    "persistent_context_color", "persistent_context_color_shape"
  )
  codes <- c("A", "B", "C", "D")
  datasets <- c("baron_human", "darmanis_brain")
  tasks <- c(
    "label_lookup", "cross_panel_tracking",
    "related_pair_judgment", "dense_rare_detection"
  )
  set.seed(seed)
  trial_rows <- list()
  key_rows <- list()
  for (p in participants) {
    key <- data.frame(
      participant_id = p, condition_code = codes,
      condition_id = sample(condition_ids), stringsAsFactors = FALSE
    )
    base <- expand.grid(
      dataset_id = datasets, task = tasks, condition_id = condition_ids,
      KEEP.OUT.ATTRS = FALSE, stringsAsFactors = FALSE
    )
    base <- merge(base, key, by = "condition_id", sort = FALSE)
    base <- base[sample.int(nrow(base)), ]
    base$trial_order <- seq_len(nrow(base))
    base$trial_id <- sprintf("%s-T%03d", p, base$trial_order)
    trial_rows[[p]] <- base[, c(
      "participant_id", "trial_id", "trial_order", "dataset_id",
      "task", "condition_code"
    )]
    key_rows[[p]] <- key
  }
  trials <- do.call(rbind, trial_rows)
  rownames(trials) <- NULL
  key <- do.call(rbind, key_rows)
  rownames(key) <- NULL
  responses <- transform(
    trials, selected_response = NA_character_, correct = NA,
    response_ms = NA_real_, confidence_1_to_5 = NA_integer_,
    technical_issue = NA_character_
  )
  list(trials = trials, condition_key = key, blank_responses = responses)
}

pilot <- make_pilot_manifest()
progression_criteria_draft <- data.frame(
  domain = c(
    "completion", "primary-outcome missingness", "technical reliability",
    "session burden", "task comprehension"
  ),
  example_green = c(
    ">= 80% complete all trials", "<= 5% missing accuracy fields",
    "<= 5% trials with a technical failure", "median session <= 45 minutes",
    ">= 80% pass pre-specified comprehension check"
  ),
  status = "DRAFT: justify and freeze before recruitment",
  stringsAsFactors = FALSE
)
head(pilot$trials, 8)
progression_criteria_draft

## Optional artifact export

Outputs go to `SCCHROMATIC_VALIDATION_DIR` when set, otherwise to the user's platform-specific data directory, never into the package source tree. The participant condition key is placed in a separate `restricted` directory. Do not place participant identifiers, donor identifiers, consent records, or raw response exports in Git.

In [ ]:
if (WRITE_OUTPUTS) {
  output_root <- Sys.getenv(
    "SCCHROMATIC_VALIDATION_DIR",
    unset = file.path(tools::R_user_dir("scChromatic-validation", "data"),
                      format(Sys.Date(), "%Y%m%d"))
  )
  dir.create(output_root, recursive = TRUE, showWarnings = FALSE)
  restricted <- file.path(output_root, "restricted")
  dir.create(restricted, showWarnings = FALSE)
  benchmark_results <- smoke_benchmark
  assignment_results <- smoke_assignment_sensitivity
  persistence_results <- smoke_persistence
  ablation_results <- smoke_ablations$scores
  relationship_results <- smoke_relationship_summary
  label_support_results <- NULL
  prepared_manifest <- NULL
  if (length(real_results)) {
    benchmark_results <- rbind(
      benchmark_results, do.call(rbind, lapply(real_results, `[[`, "benchmark"))
    )
    assignment_results <- rbind(
      assignment_results, do.call(rbind, lapply(real_results, `[[`, "assignment_sensitivity"))
    )
    persistence_results <- rbind(
      persistence_results, do.call(rbind, lapply(real_results, `[[`, "persistence"))
    )
    ablation_results <- rbind(
      ablation_results, do.call(rbind, lapply(real_results, function(x) x$ablations$scores))
    )
    relationship_results <- rbind(
      relationship_results, do.call(rbind, lapply(real_results, `[[`, "relationship_summary"))
    )
    label_support_results <- do.call(rbind, lapply(real_results, `[[`, "label_support"))
    prepared_dir <- file.path(output_root, "prepared")
    dir.create(prepared_dir, showWarnings = FALSE)
    prepared_manifest <- do.call(rbind, lapply(names(real_results), function(id) {
      path <- file.path(prepared_dir, paste0(id, ".rds"))
      saveRDS(real_results[[id]]$prepared, path, version = 3)
      data.frame(
        dataset_id = id, file = basename(path), md5 = unname(tools::md5sum(path)),
        stringsAsFactors = FALSE
      )
    }))
  }
  utils::write.csv(benchmark_results, file.path(output_root, "benchmark.csv"), row.names = FALSE)
  utils::write.csv(assignment_results, file.path(output_root, "assignment_sensitivity.csv"), row.names = FALSE)
  utils::write.csv(persistence_results, file.path(output_root, "persistence.csv"), row.names = FALSE)
  utils::write.csv(ablation_results, file.path(output_root, "ablations.csv"), row.names = FALSE)
  utils::write.csv(relationship_results, file.path(output_root, "relationship_support.csv"), row.names = FALSE)
  if (!is.null(label_support_results)) {
    utils::write.csv(label_support_results, file.path(output_root, "label_support.csv"), row.names = FALSE)
    utils::write.csv(prepared_manifest, file.path(output_root, "prepared_manifest.csv"), row.names = FALSE)
  }
  utils::write.csv(dataset_manifest, file.path(output_root, "dataset_manifest.csv"), row.names = FALSE)
  utils::write.csv(comparator_manifest, file.path(output_root, "comparator_manifest.csv"), row.names = FALSE)
  utils::write.csv(package_versions, file.path(output_root, "package_versions.csv"), row.names = FALSE)
  utils::write.csv(pilot$trials, file.path(output_root, "pilot_trials.csv"), row.names = FALSE)
  utils::write.csv(pilot$blank_responses, file.path(output_root, "pilot_blank_responses.csv"), row.names = FALSE)
  utils::write.csv(pilot$condition_key, file.path(restricted, "pilot_condition_key.csv"), row.names = FALSE)
  utils::write.csv(progression_criteria_draft, file.path(output_root, "progression_criteria_DRAFT.csv"), row.names = FALSE)
  saveRDS(smoke_ablations$encodings, file.path(output_root, "smoke_redundant_encodings.rds"))
  write_sc_color_map(smoke_primary_map, file.path(output_root, "smoke_relationship_map.json"))
  capture.output(sessionInfo(), file = file.path(output_root, "sessionInfo.txt"))
  message("Validation artifacts written to: ", output_root)
} else {
  message("WRITE_OUTPUTS is FALSE; no validation artifacts were written.")
}

## Interpretation checklist

- Keep confirmatory Baron/Segerstolpe results separate from exploratory Darmanis and scale-stress Zilionis results.
- Report all eligible comparators, failed/over-capacity methods, runtime controls, label exclusions, zero-filled pairs, and assignment permutations.
- Present the stability-versus-fit trade-off instead of optimizing one endpoint post hoc.
- Verify that persistent-map recoloring is exactly zero; treat any nonzero result as an implementation failure.
- Treat shape/pattern as redundant channels, not evidence that the color palette itself improved.
- Treat participant-pilot data as feasibility evidence; define a powered confirmatory study only after the pilot and ethics review.
- Preserve named maps, optimizer provenance, prepared-data checksums, package versions, seeds, and the unedited analysis outputs.

### Primary resources

- [scRNAseq data package vignette](https://bioconductor.org/packages/release/data/experiment/vignettes/scRNAseq/inst/doc/scRNAseq.html) and [Segerstolpe OSCA workflow](https://bioconductor.org/books/release/OSCA.workflows/segerstolpe-human-pancreas-smart-seq2.html)
- [ggplot2 hue scale](https://ggplot2.tidyverse.org/reference/scale_hue.html), [colorspace HCL palettes](https://colorspace.r-forge.r-project.org/articles/hcl_palettes.html), and [Polychrome manual](https://r-forge.r-universe.dev/Polychrome/doc/manual.html)
- [scatterHatch redundant pattern encoding](https://elifesciences.org/articles/82128)
- [Szafir, Modeling Color Difference for Visualization Design](https://pubmed.ncbi.nlm.nih.gov/28866544/)
- [BMJ tutorial on pilot sample-size justification](https://www.bmj.com/content/390/bmj-2024-083405) and [CONSORT pilot/feasibility extension](https://www.bmj.com/content/355/bmj.i5239)